# 10d Evaluate and Refine Concepts

Build yearly abstract-prevalence signals and score them with fixed direct forecasts. Run this notebook during discovery rounds; edit `CURRENT_POPULATION`, `CANDIDATE_IDS`, and `PARENT_CHILD_REPLACEMENTS` after reviewing the development results. Confirmation pairs are not used to choose concepts.

In [2]:
from pathlib import Path
import json
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "amr").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "amr").exists():
    raise RuntimeError(f"Could not find repository root from {Path.cwd()}")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from amr.pipeline_config import cfg, cfg_path, load_pipeline_config
from amr.pipelines.concepts import build_si_concept_paths, load_concept_corpus, corpus_preflight
from amr.si.concepts import (
    SEED_CONCEPTS, concept_catalog_hash, estimate_classification_cost,
    classify_concept_abstracts, build_yearly_concept_signals,
    rolling_origin_concept_forecasts, summarise_incremental_forecast_gains,
    propose_concept_children,
)

PIPELINE_CONFIG = load_pipeline_config(project_root=PROJECT_ROOT)
CONCEPT_PATHS = build_si_concept_paths(PROJECT_ROOT)
MODEL_NAME = str(cfg(PIPELINE_CONFIG, "system1.model", "typesafe/jev-1.13"))

corpus = pd.read_parquet(CONCEPT_PATHS["corpus"])
si_df = pd.read_parquet(PROJECT_ROOT / "outputs" / "parquets" / "yearly_si.parquet")
catalog = json.loads(CONCEPT_PATHS["catalog"].read_text())
CATALOG_VERSION = concept_catalog_hash(catalog)
scores = pd.read_parquet(CONCEPT_PATHS["scores"])
valid = scores.loc[
    scores["catalog_version"].astype(str).eq(CATALOG_VERSION)
    & scores["model_name"].astype(str).eq(MODEL_NAME)
    & scores["status"].eq("ok")
]
expected_pmids = set(corpus.loc[corpus["abstract_available"], "pmid"].astype(str))
scored_pmids = set(valid["pmid"].astype(str))
missing_pmids = expected_pmids - scored_pmids
if missing_pmids:
    raise RuntimeError(f"Catalog scores are incomplete for {len(missing_pmids):,} abstracts. Resume notebook 10c first.")
signals = build_yearly_concept_signals(corpus, scores, catalog_version=CATALOG_VERSION, model_name=MODEL_NAME)
signals.to_parquet(CONCEPT_PATHS["signals"], index=False)


## Configure the discovery round

`CURRENT_POPULATION` lists concepts retained from the previous development round. `CANDIDATE_IDS` lists concepts to test. Each candidate is compared with the SI-only baseline and, when a population is specified, with that population. These conditional gains measure whether each selected concept contributes beyond the other population members. `PARENT_CHILD_REPLACEMENTS` tests whether child signals predict better than their parent.

Start with an empty population to inspect seed concepts individually. After each round, use development results and scientific interpretability to decide what to retain, add, or refine. Do not use notebook 10e confirmation results for these choices.

In [3]:
CONCEPT_IDS = [str(concept["concept_id"]) for concept in catalog]
CURRENT_POPULATION = []  # concept IDs that currently survive the development round
CANDIDATE_IDS = CONCEPT_IDS  # initially compare all seed concepts
PARENT_CHILD_REPLACEMENTS = {}  # example: {"surveillance": ["genomic_surveillance", "hospital_surveillance"]}
model_sets = {f"concept:{concept_id}": [concept_id] for concept_id in CANDIDATE_IDS}
comparisons = {f"concept:{concept_id}": "baseline" for concept_id in CANDIDATE_IDS}
if CURRENT_POPULATION:
    model_sets["population"] = list(CURRENT_POPULATION)
    comparisons["population"] = "baseline"
    for candidate in CANDIDATE_IDS:
        if candidate not in CURRENT_POPULATION:
            name = f"population_plus:{candidate}"
            model_sets[name] = [*CURRENT_POPULATION, candidate]
            comparisons[name] = "population"
for parent_id, child_ids in PARENT_CHILD_REPLACEMENTS.items():
    # Hold the other surviving concepts constant while replacing this parent.
    background = [concept_id for concept_id in CURRENT_POPULATION if concept_id != parent_id]
    model_sets[f"parent:{parent_id}"] = [*background, parent_id]
    model_sets[f"children:{parent_id}"] = [*background, *child_ids]
    comparisons[f"parent:{parent_id}"] = "baseline"
    comparisons[f"children:{parent_id}"] = f"parent:{parent_id}"


## Development rolling-origin forecasts

At every eligible origin, the model uses three prior annual SI values and three prior annual values for each selected concept to forecast SI directly one, two, and three years ahead. It fits pair-specific intercepts and shared lag slopes. A pair needs at least ten prior observed SI values; missing SI is not smoothed or interpolated.

Forecast rows are saved so later review does not require recomputing Jev labels. The forecast cell prints progress every ten model-origin steps with elapsed time and ETA. A lower error means a concept adds predictive information under this setup; it does not show that the concept causes interest to change.

In [4]:
split = pd.read_csv(CONCEPT_PATHS["pair_split"])
development_pairs = set(map(tuple, split.loc[split["split"].eq("development"), ["pathogen", "antibiotic"]].astype(str).to_numpy()))
forecasts = rolling_origin_concept_forecasts(
    si_df, signals, concept_ids=CONCEPT_IDS, model_sets=model_sets,
    development_pairs=development_pairs,
    fit_start_year=int(cfg(PIPELINE_CONFIG, "si.fit_start_year", 2000)),
    fit_end_year=int(cfg(PIPELINE_CONFIG, "si.fit_end_year", 2023)),
    lag_length=3, horizons=(1, 2, 3), min_prior_observations=10,
    progress_every=10,  # ETA update every ten model-origin steps
)
forecasts = forecasts.merge(split, on=["pathogen", "antibiotic"], how="left", validate="many_to_one")
forecasts = forecasts.loc[forecasts["split"].eq("development")].drop(columns="split")
forecasts.to_parquet(CONCEPT_PATHS["forecasts"], index=False)
pair_summary, population_summary = summarise_incremental_forecast_gains(forecasts, comparisons)
pair_summary.to_parquet(CONCEPT_PATHS["pair_summary"], index=False)
population_summary.to_parquet(CONCEPT_PATHS["population_summary"], index=False)
population_summary.sort_values("generality", ascending=False)


,model,reference_model,horizon,generality,specialist_strength,pairs
3,concept:diagnostics,baseline,1,0.000561,0.005372,74
5,concept:diagnostics,baseline,3,0.000510,0.007949,74
2,concept:agricultural_use,baseline,3,0.000360,0.002042,74
28,concept:diagnostics,baseline,0,0.000137,0.004452,77
21,concept:surveillance,baseline,1,0.000041,0.001599,74
6,concept:genomic_evidence,baseline,1,-0.000002,0.001794,74
11,concept:hospital_transmission,baseline,3,-0.000033,0.004147,74
30,concept:hospital_transmission,baseline,0,-0.000044,0.002371,77
18,concept:policy_guideline_change,baseline,1,-0.000059,0.001361,74
10,concept:hospital_transmission,baseline,2,-0.000102,0.002626,76


## Horizon-specific and concept-level results

A positive `forecast_gain` means lower mean squared error than the reference model. `generality` averages gain across eligible pairs; `specialist_strength` is the 90th percentile of pair-level gain. Read each horizon alongside the equal-weight aggregate: a concept may help at one horizon and hurt at another. Check pair counts before interpreting a strong specialist score as broad evidence.

In [5]:
pair_summary.groupby(["model", "horizon"], as_index=False).agg(
    generality=("forecast_gain", "mean"),
    specialist_strength=("forecast_gain", lambda values: values.quantile(0.9)),
    pairs=("forecast_gain", "count"),
    forecasts=("forecasts", "sum"),
).sort_values(["model", "horizon"])


,model,horizon,generality,specialist_strength,pairs,forecasts
0,concept:agricultural_use,1,-0.000187,0.000515,74,725
1,concept:agricultural_use,2,-0.000512,0.000324,76,729
2,concept:agricultural_use,3,0.000360,0.002042,74,718
3,concept:diagnostics,1,0.000561,0.005372,74,725
4,concept:diagnostics,2,-0.000462,0.004112,76,729
5,concept:diagnostics,3,0.000510,0.007949,74,718
6,concept:genomic_evidence,1,-0.000002,0.001794,74,725
7,concept:genomic_evidence,2,-0.000154,0.001704,76,729
8,concept:genomic_evidence,3,-0.000697,0.001744,74,718
9,concept:hospital_transmission,1,-0.000183,0.001283,74,725
